In [ ]:
# ==========================================
# SECTION 1 : IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from scipy.optimize import curve_fit
from collections import deque 
from datetime import datetime, timedelta # Date and time handling
import matplotlib.pyplot as plt # Plotting
from IPython.display import clear_output # Clear previous output during live monitoring
import time # Used for pause timings
import warnings # Suppress unnecessary warning

warnings.filterwarnings("ignore")

# ==========================================
# SECTION 2 : LIVE DATA FILE
# ==========================================

# This CSV file is continuously updated
# by the replay/writer notebook.
LIVE_FILE = "live_data.csv"

# ==========================================
# SECTION 3 : SYSTEM PARAMETERS
# ==========================================

# --------------------------------------------------
# DATA ACQUISITION SETTINGS
# Real ICU sampling interval.
# Meaning:
# One row in the original dataset represents
# approximately 5 seconds of real physiological data.
REAL_SAMPLE_INTERVAL = 5.0      # seconds

# Replay speed.
# During testing we do not want to wait
# 5 real seconds for every sample.
# Therefore:
# 1 dataset sample
# 0.05 seconds in replay mode
REPLAY_INTERVAL = 0.01          # seconds

# --------------------------------------------------
# PRx SETTINGS
# --------------------------------------------------

# PRx window size.
# PRx is calculated using the latest
# 60 samples of MAP and ICP.
# Since:
# 60 samples * 5 sec/sample
# 300 seconds
# 5 minutes
# Physiologically this corresponds to
# approximately a 5-minute PRx window.
PRX_WINDOW_SAMPLES = 60

# --------------------------------------------------
# CPPopt SETTINGS
# --------------------------------------------------

CPP_BIN_WIDTH = 5 # CPP Bin width = 5

CPP_MIN = 40        # Physiological CPP range accepted
CPP_MAX = 120       # for CPPopt.

# Number of historical PRx values
# used for CPPopt calculation.
# 240 PRx points approximately represent
# several hours of physiological behavior.
CPP_OPT_HISTORY = 240

# --------------------------------------------------
# CPPopt QUALITY CONTROL
# --------------------------------------------------

# Minimum acceptable R^2.
# If fitted quadratic curve is poor,
# CPPopt should not be trusted.
MIN_R2 = 0.20

# Minimum number of CPP bins required.
# Example:
# If only 2 bins exist,
# curve fitting becomes unreliable.
MIN_BINS_REQUIRED = 5   

# Minimum number of PRx values inside each CPP bin.
# This prevents a bin from being
# created using only one random value.
MIN_VALUES_PER_BIN = 3

# Maximum acceptable PRx minimum. After fitting the U-curve,
# If minimum PRx itself is high autoregulation is poor and CPPopt is physiologically meaningless.
MAX_ALLOWED_MIN_PRX = 0.25

# --------------------------------------------------
# GAP DETECTION SETTINGS
# --------------------------------------------------

# If timestamp difference exceeds this,
# a data gap is declared.
MAX_ALLOWED_GAP_SEC = 10

# --------------------------------------------------
# GRAPH DISPLAY SETTINGS
# --------------------------------------------------

# If TRUE:
# show entire graph history.
SHOW_ALL_GRAPH_POINTS = True

# If FALSE:
# show only latest N points.
GRAPH_POINTS_TO_SHOW = 100

# ==========================================
# SECTION 4 : DATA BUFFERS
# ==========================================

# --------------------------------------------------
# MAP BUFFER
# --------------------------------------------------

# Stores latest 60 MAP samples.
# deque(maxlen=60)
# automatically removes oldest value
# when a new value arrives.
map_buffer = deque(maxlen=PRX_WINDOW_SAMPLES)

# --------------------------------------------------
# ICP BUFFER
# --------------------------------------------------

# Stores latest 60 ICP samples.
icp_buffer = deque(maxlen=PRX_WINDOW_SAMPLES)

# --------------------------------------------------
# CPPopt HISTORY STORAGE
# --------------------------------------------------

# Stores historical PRx values.
# These values are used to generate
# CPPopt U-shaped curves.
prx_history = deque(maxlen=CPP_OPT_HISTORY)
mean_cpp_history = deque(maxlen=CPP_OPT_HISTORY)   # Stores corresponding mean CPP values.
prx_time_history = deque(maxlen=CPP_OPT_HISTORY)    # Stores timestamps of PRx calculations.
cppopt_time_history = deque(maxlen=CPP_OPT_HISTORY)  # Stores timestamps of CPPopt calculations.

# --------------------------------------------------
# GRAPH DATA STORAGE
# --------------------------------------------------

cpp_series = []     # CPP graph values
prx_series = []     # PRx graph values
cppopt_series = []  # CPPopt graph values
time_series = []    # Common graph time axis

# ==========================================
# SECTION 5 : STATUS VARIABLES
# ==========================================

last_processed_row = 0            # Last processed CSV row prevents reprocessing same rows.
last_timestamp = None             # Previous timestamp used for gap detection.
current_clinical_minute = None    # Current clinical minute used to aggregate samples inside a minute.
samples_in_current_minute = 0     # Number of samples collected inside current minute.
current_prx = np.nan              # Current PRx value.
current_cppopt = np.nan           # Current CPPopt value.

# Last valid CPPopt.
# During gaps we continue displaying
# this value instead of showing
# CPPopt unavailable.
last_valid_cppopt = np.nan

latest_map = np.nan    # Latest MAP value.
latest_icp = np.nan   # Latest ICP value
latest_cpp = np.nan   # Latest CPP value.

# Gap status flag.
# TRUE when replaying a detected gap.
gap_active = False

gap_reason = ""       # Gap reason text.

missing_samples = 0   # Number of missing samples corresponding to detected gap.

In [2]:
# ==========================================
# SECTION 6 : CPP CALCULATION
# ==========================================

def calculate_cpp(map_value, icp_value):
    return map_value - icp_value  # CPP = MAP - ICP

# ==========================================
# SECTION 7 : PRX CALCULATION
# ==========================================

def calculate_prx(map_window, icp_window):
    # PRx requires a complete 60-sample window.
    if len(map_window) < PRX_WINDOW_SAMPLES:
        return np.nan, f"Need {PRX_WINDOW_SAMPLES - len(map_window)} more samples"

    try:
        # Pearson correlation between MAP and ICP.
        prx, _ = pearsonr(map_window, icp_window)
        return prx, "Valid"
    except Exception as e:
        return np.nan, f"Correlation error : {str(e)}"

# ==========================================
# SECTION 8 : QUADRATIC FUNCTION
# ==========================================

def quadratic(x, a, b, c):
    # CPPopt assumes a U-shaped relationship:
    # PRx = a*x² + b*x + c
    # Minimum point of this curve
    # becomes CPPopt.
    return a*x**2 + b*x + c

# ==========================================
# SECTION 9 : R² CALCULATION
# ==========================================

def calculate_r2(y_actual, y_pred):
    ss_res = np.sum((y_actual - y_pred)**2)  # residual error
    ss_tot = np.sum((y_actual - np.mean(y_actual))**2)  # total variation

    if ss_tot == 0:
        return 0

    return 1 - (ss_res / ss_tot)

# ==========================================
# SECTION 10 : CPPopt CALCULATION
# ==========================================

def calculate_cppopt(prx_hist, cpp_hist):
    # --------------------------------------------------
    # STEP 1
    # Need enough historical PRx values.
    # --------------------------------------------------
    if len(prx_hist) < CPP_OPT_HISTORY:
        return (np.nan, f"Need {CPP_OPT_HISTORY - len(prx_hist)} more PRx values")

    # --------------------------------------------------
    # STEP 2
    # Use latest CPP_OPT_HISTORY points.
    # --------------------------------------------------
    prx_hist = np.array(prx_hist)[-CPP_OPT_HISTORY:]
    cpp_hist = np.array(cpp_hist)[-CPP_OPT_HISTORY:]

    # --------------------------------------------------
    # STEP 3
    # Create CPP bins.
    # --------------------------------------------------
    bins = np.arange(cpp_hist.min(), cpp_hist.max() + CPP_BIN_WIDTH, CPP_BIN_WIDTH)
    bin_centers = []
    bin_prx = []

    # --------------------------------------------------
    # STEP 4
    # Average PRx inside each CPP bin.
    # --------------------------------------------------
    for i in range(len(bins)-1):
        mask = ((cpp_hist >= bins[i]) & (cpp_hist < bins[i+1]))
        # Require minimum number of values per bin.
        if np.sum(mask) >= MIN_VALUES_PER_BIN:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            bin_prx.append(np.mean(prx_hist[mask]))

    # --------------------------------------------------
    # STEP 5
    # Require sufficient bins.
    # --------------------------------------------------
    if len(bin_centers) < MIN_BINS_REQUIRED:
        return (np.nan, f"Need at least {MIN_BINS_REQUIRED} CPP bins")

    bin_centers = np.array(bin_centers)
    bin_prx = np.array(bin_prx)

    try:
        # --------------------------------------------------
        # STEP 6
        # Fit quadratic curve.
        # --------------------------------------------------
        popt, _ = curve_fit(quadratic, bin_centers, bin_prx)
        a, b, c = popt

        # --------------------------------------------------
        # STEP 7
        # U-shape validation.
        # --------------------------------------------------
        if a <= 0:
            return (np.nan, "Quadratic coefficient a <= 0")

        # --------------------------------------------------
        # STEP 8
        # Calculate CPPopt.
        # --------------------------------------------------
        cppopt = -b / (2*a)

        # --------------------------------------------------
        # STEP 9
        # CPPopt physiological range.
        # --------------------------------------------------
        if cppopt < CPP_MIN or cppopt > CPP_MAX:
            return (np.nan, f"CPPopt outside range ({cppopt:.1f})")

        # --------------------------------------------------
        # STEP 10
        # Calculate fitted values.
        # --------------------------------------------------
        fitted = quadratic(bin_centers, *popt)

        # --------------------------------------------------
        # STEP 11
        # Calculate R².
        # --------------------------------------------------
        r2 = calculate_r2(bin_prx, fitted)
        if r2 < MIN_R2:
            return (np.nan, f"Poor fit R²={r2:.2f}")

        # --------------------------------------------------
        # STEP 12
        # Calculate minimum predicted PRx.
        # --------------------------------------------------
        min_prx = quadratic(cppopt, *popt)
        if min_prx > MAX_ALLOWED_MIN_PRX:
            return (np.nan, f"Minimum PRx too high ({min_prx:.2f})")

        # --------------------------------------------------
        # STEP 13
        # Success.
        # --------------------------------------------------
        return cppopt, "Valid"

    except Exception as e:
        return (np.nan, f"Curve fitting error : {str(e)}")

# ==========================================
# SECTION 11 : GAP DETECTION
# ==========================================

def check_gap(prev_time, current_time):
    if prev_time is None:
        return False, 0

    gap_sec = (current_time - prev_time).total_seconds()

    if gap_sec > MAX_ALLOWED_GAP_SEC:
        return True, gap_sec

    return False, gap_sec

# ==========================================
# SECTION 11A : INSERT NaN POINTS
# ==========================================

def insert_gap_nan_points(previous_timestamp, current_timestamp):
    global time_series
    global cpp_series
    global prx_series
    global cppopt_series

    if previous_timestamp is None:
        return

    gap_sec = (current_timestamp - previous_timestamp).total_seconds()
    missing_samples = int(gap_sec / REAL_SAMPLE_INTERVAL) - 1

    if missing_samples <= 0:
        return

    for i in range(1, missing_samples + 1):
        missing_time = previous_timestamp + timedelta(seconds=i*REAL_SAMPLE_INTERVAL)
        time_series.append(missing_time)
        cpp_series.append(np.nan)  # break CPP graph
        prx_series.append(np.nan)  # break PRx graph
        cppopt_series.append(np.nan)  # break CPPopt graph

In [3]:
# ==========================================
# SECTION 12 : LIVE PLOTTING
# ==========================================

def plot_live():
    # ------------------------------------------
    # Clear previous output so that
    # only latest graphs remain visible.
    # ------------------------------------------
    clear_output(wait=True)

    # ------------------------------------------
    # Select graph range.
    # ------------------------------------------
    if SHOW_ALL_GRAPH_POINTS:
        plot_time = time_series
        plot_cpp = cpp_series
        plot_prx = prx_series
        plot_cppopt = cppopt_series
    else:
        plot_time = time_series[-GRAPH_POINTS_TO_SHOW:]
        plot_cpp = cpp_series[-GRAPH_POINTS_TO_SHOW:]
        plot_prx = prx_series[-GRAPH_POINTS_TO_SHOW:]
        plot_cppopt = cppopt_series[-GRAPH_POINTS_TO_SHOW:]

    # ------------------------------------------
    # Create figure.
    # ------------------------------------------
    plt.figure(figsize=(15,10))

    # ==========================================
    # CPP GRAPH
    # ==========================================
    plt.subplot(3,1,1)
    plt.plot(plot_time, plot_cpp, linewidth=1.5)
    plt.title("Cerebral Perfusion Pressure (CPP)")
    plt.ylabel("CPP (mmHg)")
    plt.grid(True)

    # ------------------------------------------
    # Reference lines.
    # ------------------------------------------
    plt.axhline(y=CPP_MIN, linestyle="--", alpha=0.5)
    plt.axhline(y=CPP_MAX, linestyle="--", alpha=0.5)

    # ==========================================
    # PRx GRAPH
    # ==========================================
    plt.subplot(3,1,2)
    plt.plot(plot_time, plot_prx, linewidth=1.5)
    plt.title("Pressure Reactivity Index (PRx)")
    plt.ylabel("PRx")
    plt.grid(True)

    # ------------------------------------------
    # PRx reference line.
    # ------------------------------------------
    plt.axhline(y=0.25, linestyle="--", alpha=0.5)

    # ==========================================
    # CPPopt GRAPH
    # ==========================================
    plt.subplot(3,1,3)
    plt.plot(plot_time, plot_cppopt, linewidth=1.5)
    plt.title("Optimal Cerebral Perfusion Pressure (CPPopt)")
    plt.ylabel("CPPopt (mmHg)")
    plt.xlabel("Time")
    plt.grid(True)

    # ------------------------------------------
    # CPPopt physiological limits.
    # ------------------------------------------
    plt.axhline(y=CPP_MIN, linestyle="--", alpha=0.5)
    plt.axhline(y=CPP_MAX, linestyle="--", alpha=0.5)

    # ------------------------------------------
    # Improve spacing.
    # ------------------------------------------
    plt.tight_layout()
    plt.show()

# ==========================================
# SECTION 12A : STATUS DISPLAY
# ==========================================

def display_monitor_status(
    timestamp,
    map_value,
    icp_value,
    cpp_value,
    prx_value,
    cppopt_value,
    prx_reason,
    cppopt_reason,
    gap_active=False,
    gap_message=""
):
    print("="*70)
    print(f"Time : {timestamp}")
    print("-" * 70)

    # ======================================
    # GAP MODE
    # ======================================
    if gap_active:
        print("MAP     : Data Unavailable")
        print("ICP     : Data Unavailable")
        print("CPP     : Data Unavailable")
        print("PRx     : Data Unavailable")

        # ----------------------------------
        # During gap:
        # show last valid CPPopt.
        # ----------------------------------
        if not np.isnan(last_valid_cppopt):
            print(f"CPPopt  : {last_valid_cppopt:.2f} (Last Valid Value)")
        else:
            print("CPPopt  : Data Unavailable")

        print("-" * 70)
        print(f"STATUS : {gap_message}")
        print("=" * 70)
        return

    # ======================================
    # NORMAL MODE
    # ======================================
    print(f"MAP     : {map_value:.2f}")
    print(f"ICP     : {icp_value:.2f}")
    print(f"CPP     : {cpp_value:.2f}")

    # --------------------------------------
    # PRx display.
    # --------------------------------------
    if np.isnan(prx_value):
        print("PRx     : Unavailable")
        print(f"Reason  : {prx_reason}")
    else:
        print(f"PRx     : {prx_value:.4f}")

    # --------------------------------------
    # CPPopt display.
    # --------------------------------------
    if np.isnan(cppopt_value):
        print("CPPopt  : Unavailable")
        print(f"Reason  : {cppopt_reason}")
    else:
        print(f"CPPopt  : {cppopt_value:.2f}")

    print("=" * 70)

In [ ]:
# ==========================================
# SECTION 13 : MAIN LOOP
# ==========================================

while True:
    try:
        # --------------------------------------
        # Read latest CSV.
        # --------------------------------------
        df = pd.read_csv(LIVE_FILE)

        # --------------------------------------
        # CPU optimization.
        # --------------------------------------
        if len(df) <= last_processed_row:
            time.sleep(0.05)
            continue

        # --------------------------------------
        # Process only new rows.
        # --------------------------------------
        new_rows = df.iloc[last_processed_row:]

        for _, row in new_rows.iterrows():
            # ==================================
            # TIMESTAMP
            # ==================================
            timestamp = pd.to_datetime(row["TTDate"] + " " + row["TTTime"])

            # ==================================
            # CURRENT VALUES
            # ==================================
            MAP = row["mean1"]
            ICP = row["mean2"]
            CPP = calculate_cpp(MAP, ICP)

            latest_map = MAP
            latest_icp = ICP
            latest_cpp = CPP

            # ==================================
            # GAP DETECTION
            # ==================================
            gap_found, gap_sec = check_gap(last_timestamp, timestamp)

            # ==================================
            # HANDLE GAP
            # ==================================
            if gap_found:
                # ------------------------------
                # Insert NaN points in graphs.
                # ------------------------------
                insert_gap_nan_points(last_timestamp, timestamp)

                # ------------------------------
                # Missing samples.
                # ------------------------------
                missing_samples = int(round(gap_sec / REAL_SAMPLE_INTERVAL))

                # ------------------------------
                # Replay wait duration.
                # ------------------------------
                replay_gap_time = missing_samples * REPLAY_INTERVAL
                gap_message = f"Data gap detected ({gap_sec:.1f} sec, {missing_samples} samples)"

                # ------------------------------
                # Show unavailable status.
                # ------------------------------
                display_monitor_status(
                    timestamp=timestamp,
                    map_value=np.nan,
                    icp_value=np.nan,
                    cpp_value=np.nan,
                    prx_value=np.nan,
                    cppopt_value=np.nan,
                    prx_reason="Gap detected",
                    cppopt_reason="",
                    gap_active=True,
                    gap_message=gap_message
                )

                # ------------------------------
                # Simulate missing time.
                # ------------------------------
                time.sleep(replay_gap_time)

            # ==================================
            # UPDATE BUFFERS
            # ==================================
            map_buffer.append(MAP)
            icp_buffer.append(ICP)

            # ==================================
            # PRx CALCULATION
            # ==================================
            current_prx, prx_reason = calculate_prx(list(map_buffer), list(icp_buffer))

            # ==================================
            # CPP HISTORY
            # ==================================
            if not np.isnan(current_prx):
                mean_cpp = np.mean(np.array(map_buffer) - np.array(icp_buffer))
                prx_history.append(current_prx)
                mean_cpp_history.append(mean_cpp)
                prx_time_history.append(timestamp)

            # ==================================
            # CPPopt CALCULATION
            # ==================================
            current_cppopt, cppopt_reason = calculate_cppopt(prx_history, mean_cpp_history)

            # ==================================
            # STORE LAST VALID CPPopt
            # ==================================
            if not np.isnan(current_cppopt):
                last_valid_cppopt = current_cppopt

            # ==================================
            # GRAPH STORAGE
            # ==================================
            time_series.append(timestamp)
            cpp_series.append(CPP)
            prx_series.append(current_prx)

            # ----------------------------------
            # Use latest valid CPPopt
            # when current value unavailable.
            # ----------------------------------
            if np.isnan(current_cppopt):
                cppopt_series.append(last_valid_cppopt)
            else:
                cppopt_series.append(current_cppopt)
                cppopt_time_history.append(timestamp)

            # ==================================
            # UPDATE STATUS
            # ==================================
            last_timestamp = timestamp

            # ==================================
            # LIVE GRAPH
            # ==================================
            plot_live()

            # ==================================
            # MONITOR DISPLAY
            # ==================================
            display_monitor_status(
                timestamp=timestamp,
                map_value=latest_map,
                icp_value=latest_icp,
                cpp_value=latest_cpp,
                prx_value=current_prx,
                cppopt_value=(current_cppopt if not np.isnan(current_cppopt) else last_valid_cppopt),
                prx_reason=prx_reason,
                cppopt_reason=cppopt_reason,
                gap_active=False,
                gap_message=""
            )

        # ======================================
        # UPDATE PROCESSED ROW COUNT
        # ======================================
        last_processed_row = len(df)

    except Exception as e:
        print(f"Error : {str(e)}")
        time.sleep(1)